# OBJECTIVE
The objective of this methodology  is to reconcile bilateral trade data reported in UN Comtrade to establish a single, consistent trade value between partners. By addressing discrepancies and measuring the reliability of reported trade flows, this approach enhances the accuracy of trade records, resulting in a cohesive dataset for international economic analysis.

# Installation of complexity libraries

In [66]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
!pip -q install econci

In [68]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import econci

## Load Trade data

In [69]:
root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/UN comtrade/2021"
export_name = "export_2021.csv"
import_name = "import_2021.csv"

In [70]:
imports_df = pd.read_csv(f'{root_folder}/{import_name}',dtype = {'cmdCode':str})
exports_df = pd.read_csv(f'{root_folder}/{export_name}',dtype = {'cmdCode':str})

In [71]:
imports_df = imports_df[['period', 'importerISO', 'importer', 'exporterISO', 'exporter', 'cmdCode', 'import_value']]
exports_df = exports_df[['period', 'importerISO', 'importer', 'exporterISO', 'exporter', 'cmdCode', 'export_value']]

process dataframe

# Auxiliary dataframes

1.   CIF-FOB Margins from OECD



In [72]:
root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/Trasport margins"
file_name = "transport_2021.csv"
transport_df = pd.read_csv(f'{root_folder}/{file_name}',dtype = {'product':str})

## vALIDATION DATASET
Baci data

In [ ]:
# load root folder
root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/baci data/2017-2022"
# specify file names
trade_2021= "BACI_HS17_Y2021_V202401b.csv"
country_file  = "country_codes_V202401b.csv"
product_file = "product_codes_HS17_V202401b.csv"
#read the data
trade_2021 = pd.read_csv(f'{root_folder}/{trade_2021}',dtype = {'k':str})
country_data = pd.read_csv(f'{root_folder}/{country_file}')
product_data = pd.read_csv(f'{root_folder}/{product_file}',dtype = {'code':str})

In [ ]:
# rename columns as per the documentation and to fit with the other dataframes
trade_2021 = trade_2021.rename(columns={
    't': 'period',
    'i': 'exporter',
    'j': 'importer',
    'k': 'cmdCode',
    'v': 'value',
    'q': 'quantity'
})

In [ ]:
# process Auxiliary dataframes
country_data = country_data.rename(columns={'country_code': 'exporter'})
product_data = product_data.rename(columns={'code': 'cmdCode'})

In [ ]:
trade_2021['HS4_code'] = trade_2021['cmdCode'].astype(str).str[:4]

In [ ]:
grouped_df_2021 = trade_2021.groupby(['exporter', 'importer', 'HS4_code', 'period'], as_index=False)[['value']].sum()
processed_df_2021 = pd.merge(grouped_df_2021, country_data, on='exporter', how='inner')

In [ ]:
#generate HS4 code from HS6 code
product_data['HS4_code'] = product_data['cmdCode'].astype(str).str[:4]
product_data = product_data.drop('cmdCode', axis=1)
product_data = product_data.groupby('HS4_code').agg({
    'description': 'first',   # Keep the first description
}).reset_index()
product_data.head()

In [ ]:
Baci_2021 = pd.merge(processed_df_2021, product_data, on='HS4_code', how='inner')
Baci_2021['value'] = Baci_2021['value'] * 1000

# Processing of Dataframes

In [73]:
imports_df.reset_index(drop=True, inplace=True)
exports_df.reset_index(drop=True, inplace=True)

In [74]:
exports_df.head()

,period,importerISO,importer,exporterISO,exporter,cmdCode,export_value
0,2021,LUX,Luxembourg,AUS,Australia,101,288875.443
1,2021,IDN,Indonesia,USA,USA,101,152000.000
2,2021,HKG,"China, Hong Kong SAR",USA,USA,101,294000.000
3,2021,HND,Honduras,USA,USA,101,22245.000
4,2021,HTI,Haiti,USA,USA,101,284841.000


In [75]:
imports_df.head()

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value
0,2021,AND,Andorra,FRA,France,101,591.27
1,2021,KGZ,Kyrgyzstan,DEU,Germany,101,4600.00
2,2021,KGZ,Kyrgyzstan,FRA,France,101,1300.00
3,2021,KGZ,Kyrgyzstan,BLR,Belarus,101,13185.00
4,2021,KGZ,Kyrgyzstan,BEL,Belgium,101,800.00


Processing Transport and insurance margins dataset

In [76]:
# rename columns, minus margin by 100%
transport_df.rename(columns={'reporter': 'importerISO', 'partner': 'exporterISO', 'product':'cmdCode', 'year':'period'}, inplace=True)
transport_df['new_margin'] = 100 - transport_df['margin']
transport_df = transport_df[['period','cmdCode', 'importerISO', 'exporterISO', 'new_margin']]
transport_df.dropna(inplace=True) # dropnull values

# merge imports and exports data

In [77]:
trade_data = pd.merge(imports_df, exports_df, on=['period','exporter','exporterISO', 'importer','importerISO', 'cmdCode'], how='outer')
print(trade_data.shape)
trade_data.head()

(4895397, 8)


,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value
0,2021,ALB,Albania,AFG,Afghanistan,9999,312.741,NaN
1,2021,AND,Andorra,AFG,Afghanistan,4202,598.081,NaN
2,2021,AND,Andorra,AFG,Afghanistan,6210,199.849,NaN
3,2021,AND,Andorra,AFG,Afghanistan,6403,167.117,NaN
4,2021,AND,Andorra,AFG,Afghanistan,8708,461.344,NaN


In [78]:
print(trade_data['importerISO'].nunique())
trade_data['exporterISO'].nunique()

244


244

In [79]:
trade_data['exporterISO'] = trade_data['exporterISO'].replace('S19', 'TWN')
trade_data['importerISO'] = trade_data['importerISO'].replace('S19', 'TWN')

In [80]:
trade_data.fillna(0, inplace=True)# fill nan  with zero

In [81]:
#exporter_ISO = ['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATF', 'ATG', 'TWN', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BES', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS',  'BIH', 'BLM', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF',  'CAN', 'CCK', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM',  'CPV', 'CRI', 'CUB', 'CUW', 'CXR', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK',  'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FLK', 'FRA',  'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD',  'GRL', 'GTM', 'GUM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IOT',  'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ',  'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LKA', 'LSO',  'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI',  'MLT', 'MMR', 'MNE', 'MNG', 'MNP', 'MOZ', 'MRT', 'MSR', 'MUS', 'MWI', 'MYS', 'NAM',  'NCL', 'NER', 'NFK', 'NGA', 'NIC', 'NIU', 'NLD', 'NOR', 'NPL', 'NRU', 'NZL', 'OMN',  'PAK', 'PAN', 'PCN', 'PER', 'PHL', 'PLW', 'PNG', 'POL', 'PRK', 'PRT', 'PRY', 'PSE',  'PYF', 'QAT', 'ROU', 'RUS', 'RWA', 'SAU', 'SDN', 'SEN', 'SGP', 'SHN', 'SLB',  'SLE', 'SLV', 'SMR', 'SOM', 'SPM', 'SRB', 'SSD', 'STP', 'SUR', 'SVK', 'SVN', 'SWE',  'SWZ', 'SXM', 'SYC', 'SYR', 'TCA', 'TCD', 'TGO', 'THA', 'TJK', 'TKL', 'TKM', 'TLS',  'TON', 'TTO', 'TUN', 'TUR', 'TUV', 'TZA', 'UGA', 'UKR', 'URY', 'USA', 'UZB', 'VCT',  'VEN', 'VGB', 'VNM', 'VUT', 'WLF', 'WSM', 'YEM', 'ZAF', 'ZMB', 'ZWE']
#exporter_ISO = pd.DataFrame(exporter_ISO, columns=['exporterISO'])

In [82]:
#importer_ISO = ['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATF', 'ATG', 'TWN', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BES', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS',  'BIH', 'BLM', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF',  'CAN', 'CCK', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM',  'CPV', 'CRI', 'CUB', 'CUW', 'CXR', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK',  'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FLK', 'FRA',  'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD',  'GRL', 'GTM', 'GUM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IOT',  'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ',  'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LKA', 'LSO',  'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI',  'MLT', 'MMR', 'MNE', 'MNG', 'MNP', 'MOZ', 'MRT', 'MSR', 'MUS', 'MWI', 'MYS', 'NAM',  'NCL', 'NER', 'NFK', 'NGA', 'NIC', 'NIU', 'NLD', 'NOR', 'NPL', 'NRU', 'NZL', 'OMN',  'PAK', 'PAN', 'PCN', 'PER', 'PHL', 'PLW', 'PNG', 'POL', 'PRK', 'PRT', 'PRY', 'PSE',  'PYF', 'QAT', 'ROU', 'RUS', 'RWA', 'SAU', 'SDN', 'SEN', 'SGP', 'SHN', 'SLB',  'SLE', 'SLV', 'SMR', 'SOM', 'SPM', 'SRB', 'SSD', 'STP', 'SUR', 'SVK', 'SVN', 'SWE',  'SWZ', 'SXM', 'SYC', 'SYR', 'TCA', 'TCD', 'TGO', 'THA', 'TJK', 'TKL', 'TKM', 'TLS',  'TON', 'TTO', 'TUN', 'TUR', 'TUV', 'TZA', 'UGA', 'UKR', 'URY', 'USA', 'UZB', 'VCT',  'VEN', 'VGB', 'VNM', 'VUT', 'WLF', 'WSM', 'YEM', 'ZAF', 'ZMB', 'ZWE']
#importer_ISO = pd.DataFrame(importer_ISO, columns=['importerISO'])

In [83]:
print(trade_data['exporter'].nunique())
print(trade_data['importer'].nunique())
print(trade_data['cmdCode'].nunique())

244
245
1226


# what is the raw total values?

In [84]:
total_export_value = trade_data['export_value'].sum()
total_import_value = trade_data['import_value'].sum()

print(f"Total Export Value: {total_export_value}")
print(f"Total Import Value: {total_import_value}")

Total Export Value: 21685081852265.406
Total Import Value: 20873554774724.88


# Corrections of CIF-FOB margins
This is per the OECD Database on International Transport and Insurance Costs (ITIC) - https://data-explorer.oecd.org/vis?tm=itic&pg=0&snb=1&df[ds]=dsDisseminateFinalDMZ&df[id]=DSD_ITIC%40DF_ITIC&df[ag]=OECD.SDD.TPS&df[vs]=1.0&dq=AUS...._T.A.&pd=2016%2C&to[TIME_PERIOD]=false

In [133]:
trade_df = pd.merge(trade_data, transport_df, on=['period','cmdCode', 'importerISO', 'exporterISO'], how='left')
print(trade_df.shape)
trade_df.head()

(4895397, 9)


,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin
0,2021,ALB,Albania,AFG,Afghanistan,9999,312.741,NaN,NaN
1,2021,AND,Andorra,AFG,Afghanistan,4202,598.081,NaN,91.87
2,2021,AND,Andorra,AFG,Afghanistan,6210,199.849,NaN,92.44
3,2021,AND,Andorra,AFG,Afghanistan,6403,167.117,NaN,93.25
4,2021,AND,Andorra,AFG,Afghanistan,8708,461.344,NaN,91.77


In [134]:
print(trade_df['cmdCode'].nunique())
print(trade_df['importerISO'].nunique())
print(trade_df['exporterISO'].nunique())

1226
244
244


In [147]:
print(trade_df.isna().sum())

period              0
importerISO         0
importer            0
exporterISO         0
exporter            0
cmdCode             0
import_value        0
export_value        0
new_margin          0
new_import_value    0
dtype: int64


In [136]:
# lets take care of the empty margins by filling with tha standard margin
trade_df['new_margin'] = np.where(trade_df['new_margin'].isna(), 94, trade_df['new_margin']) # standard margin for international trade

In [144]:
# fill NAN export values and NaN import values with 0
trade_df['import_value'] = np.where(trade_df['import_value'].isna(), 0, trade_df['import_value'])
trade_df['export_value'] = np.where(trade_df['export_value'].isna(), 0, trade_df['export_value'])

### calculate new import value by mutliplying ond import value by margins

In [146]:
trade_df['new_import_value'] = (trade_df['import_value'] * trade_df['new_margin'])/100

In [90]:
# process the dataframe by renaming colums
trade_df = trade_df.drop(columns=['new_margin','import_value'])
trade_df = trade_df.rename(columns={'new_import_value': 'import_value'})

In [148]:
total_export_value = trade_df['export_value'].sum()
total_import_value = trade_df['import_value'].sum()

print(f"Total Export Value: {total_export_value}")
print(f"Total Import Value: {total_import_value}")

Total Export Value: 21685081852265.406
Total Import Value: 20873554774724.88


## Trade reconciliation
steps:


1.   Calclulate accuracy level
2.   Calculate of total trade at commodity level for both exports and imports
3.   Calculate Accurately matched exports and imports
4.   Calculate Reliability indexes based only on the accurate transcations
5.   Reconcile Trade values based on reliability indexes



In [149]:
# Replace zero values with (1e-10) easy calculations.
for col in ['export_value', 'import_value']:
    trade_df[col] = np.where(trade_df[col] == 0, 1e-10, trade_df[col])

In [152]:
trade_df.sample(n=5)

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value
3191305,2021,AUT,Austria,POL,Poland,1803,1.755187e+03,8.638300e+04,99.05,1.738513e+03
1832974,2021,BLR,Belarus,IND,India,7016,1.200000e+03,1.000000e-10,89.70,1.076400e+03
2441212,2021,DZA,Algeria,LTU,Lithuania,8422,1.000000e-10,1.025560e+05,93.48,0.000000e+00
4710558,2021,ETH,Ethiopia,GBR,United Kingdom,8211,2.059060e+03,1.000000e-10,90.89,1.871480e+03
1367731,2021,ESP,Spain,FIN,Finland,8704,5.285557e+06,1.036584e+07,93.86,4.961024e+06


###  step 1: calculate Accuracy level



In [154]:
def calculate_accuracy_level(import_value, export_value):
    accuracy_level = abs(import_value - export_value) / import_value * 100
    return accuracy_level

In [155]:
trade_df.loc[:, 'accuracy_level'] = trade_df.apply(lambda row: calculate_accuracy_level(row['import_value'], row['export_value']), axis=1)

In [156]:
trade_df.head()

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value,accuracy_level
0,2021,ALB,Albania,AFG,Afghanistan,9999,312.741,1.000000e-10,94.00,293.976540,100.0
1,2021,AND,Andorra,AFG,Afghanistan,4202,598.081,1.000000e-10,91.87,549.457015,100.0
2,2021,AND,Andorra,AFG,Afghanistan,6210,199.849,1.000000e-10,92.44,184.740416,100.0
3,2021,AND,Andorra,AFG,Afghanistan,6403,167.117,1.000000e-10,93.25,155.836602,100.0
4,2021,AND,Andorra,AFG,Afghanistan,8708,461.344,1.000000e-10,91.77,423.375389,100.0


### Step 2: Calculation of total trade at commodity level
calculate the total trade reported by the importer i for commodity s and likewise for exporter.

In [157]:
trade_df['total_imports'] = trade_df.groupby(['importer','cmdCode'])['import_value'].transform('sum')
trade_df['total_exports'] = trade_df.groupby(['exporter','cmdCode'])['export_value'].transform('sum')
trade_df.head()

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value,accuracy_level,total_imports,total_exports
0,2021,ALB,Albania,AFG,Afghanistan,9999,312.741,1.000000e-10,94.00,293.976540,100.0,2.863587e+09,3.200000e-09
1,2021,AND,Andorra,AFG,Afghanistan,4202,598.081,1.000000e-10,91.87,549.457015,100.0,7.301059e+06,4.800000e-09
2,2021,AND,Andorra,AFG,Afghanistan,6210,199.849,1.000000e-10,92.44,184.740416,100.0,1.743716e+07,1.900000e-09
3,2021,AND,Andorra,AFG,Afghanistan,6403,167.117,1.000000e-10,93.25,155.836602,100.0,7.216860e+06,4.500000e-09
4,2021,AND,Andorra,AFG,Afghanistan,8708,461.344,1.000000e-10,91.77,423.375389,100.0,2.148643e+07,4.000000e-09


In [158]:
# Reset zero value
for col in ['export_value', 'import_value']:
    trade_df[col] = np.where(trade_df[col] ==1e-10, 0, trade_df[col])

In [159]:
usa_df = trade_df[trade_df['importer'] == 'USA']
usa_df_cmdcode = usa_df[usa_df['cmdCode'] == '2106']
usa_df_cmdcode.sample(n=5)

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value,accuracy_level,total_imports,total_exports
4883866,2021,USA,USA,YEM,Yemen,2106,42694.0,0.000000e+00,94.08,4.016652e+04,100.000000,7.413250e+09,1.000000e-09
2521961,2021,USA,USA,MDG,Madagascar,2106,242309.0,0.000000e+00,98.66,2.390621e+05,100.000000,7.413250e+09,1.834480e+06
3265708,2021,USA,USA,POL,Poland,2106,18180266.0,1.523961e+07,94.16,1.711854e+07,16.175005,7.413250e+09,1.678328e+09
4061970,2021,USA,USA,SWE,Sweden,2106,58809605.0,7.018531e+07,97.97,5.761577e+07,19.343276,7.413250e+09,6.990107e+08
1746630,2021,USA,USA,GUY,Guyana,2106,33082.0,9.323294e+06,92.11,3.047183e+04,28082.376673,7.413250e+09,9.466747e+06


###  Step 3: calculately accurately matched exports and imports
A threshold level must be established. It is the difference as a percentage between reported exports and reported imports. This level as been established at 20%. if it less than 20% its considered a match.



In [160]:
# Set the threshold
threshold = 25

Add a new column, 'accurate_import', to trade_df, which flags imports as "accurate" if the discrepancy (measured by accuracy_level) is within a specified threshold.

In [161]:
trade_df['accurate_import'] = trade_df.apply(lambda row: row['import_value'] if row['accuracy_level'] <= threshold else 0, axis=1)

Add a new column, 'accurate_export', to trade_df, which flags exports as "accurate" if the discrepancy (measured by accuracy_level) is within a specified threshold.


In [162]:
trade_df['accurate_export'] = trade_df.apply(lambda row: row['export_value'] if row['accuracy_level'] <= threshold else 0, axis=1)

In [163]:
trade_df.sample(n=5)

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value,accuracy_level,total_imports,total_exports,accurate_import,accurate_export
3509255,2021,ISL,Iceland,RUS,Russian Federation,303,1749.664,0.000,94.00,1644.684160,1.000000e+02,4.223233e+06,2.411601e+09,0.000,0.000
4308512,2021,FIN,Finland,TUR,Türkiye,3704,0.000,25.000,94.00,0.000000,2.500000e+13,2.754957e+04,2.585500e+04,0.000,0.000
892344,2021,NLD,Netherlands,HKG,"China, Hong Kong SAR",9606,423384.366,516966.869,91.48,387312.018017,2.210344e+01,1.142734e+07,3.744283e+08,423384.366,516966.869
1029979,2021,BRB,Barbados,CZE,Czechia,8302,770.500,5046.000,91.83,707.550150,5.548994e+02,5.734198e+06,5.975695e+08,0.000,0.000
3714560,2021,ITA,Italy,SVK,Slovakia,3814,11577.845,798.350,95.48,11054.526406,9.310450e+01,6.148229e+07,1.877055e+07,0.000,0.000


### step 4: calculate reliability indexes based on share of accurate transactions

**IMPORTER RELIABILITY INDEX**

The importer-commodity reliability index is calculated by first grouping the data in the DataFrame based on each importer and commodity code (cmdCode). For each group, a lambda function is applied to calculate the  percentage of accurate imports (total_accurate_imports) relative to total imports (total_imports)

1.  summming  of accurate_import values in the group

2. Dividing sum of accurate_import_value by total number of exports in that group.

3. Multiplying the result by 100 to express the reliability as a percentage.


In [164]:
trade_df['total_accurate_imports'] = trade_df.groupby(['importer', 'cmdCode'])['accurate_import'].transform('sum')
trade_df['total_accurate_exports'] = trade_df.groupby(['exporter', 'cmdCode'])['accurate_export'].transform('sum')

example of specific country

In [165]:
usa_df = trade_df[trade_df['exporter'] == 'USA']
usa_df_cmdcode = usa_df[usa_df['cmdCode'] == '9017']
usa_df_cmdcode.sample(n=5)

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value,accuracy_level,total_imports,total_exports,accurate_import,accurate_export,total_accurate_imports,total_accurate_exports
4513214,2021,SVN,Slovenia,USA,USA,9017,46442.174,9990.0,93.49,43418.788473,78.489379,6.221530e+06,298758923.0,0.000,0.0,4.681727e+05,26933944.0
4460507,2021,ITA,Italy,USA,USA,9017,1062627.118,1264829.0,93.79,996637.973972,19.028489,5.413711e+07,298758923.0,1062627.118,1264829.0,2.481296e+07,26933944.0
4492627,2021,PAN,Panama,USA,USA,9017,444125.000,1458229.0,93.48,415168.050000,228.337518,1.936321e+06,298758923.0,0.000,0.0,5.333000e+03,26933944.0
4493337,2021,PNG,Papua New Guinea,USA,USA,9017,5840.216,13050.0,91.77,5359.566223,123.450639,1.173324e+06,298758923.0,0.000,0.0,6.303931e+05,26933944.0
4427417,2021,CYP,Cyprus,USA,USA,9017,18250.866,106371.0,93.38,17042.658671,482.827138,6.150983e+05,298758923.0,0.000,0.0,2.961191e+05,26933944.0


In [166]:
total_cmdcode = usa_df_cmdcode['accurate_export'].sum()
total_cmdcode

26933944.0

In [167]:
trade_df['importer_commodity_reliability'] = (
    trade_df.groupby(['importer', 'cmdCode'])['total_accurate_imports'].transform(lambda x: (x / trade_df.loc[x.index, 'total_imports']) * 100))


**EXPORTER RELIABILITY INDEX**

The exporter-commodity reliability index is calculated by first grouping the data in the DataFrame based on each exporter and commodity code (cmdCode). For each group, a lambda function is applied to calculate the  percentage of accurate exports (total_accurate_exports) relative to total exports (total_exports)

1. Summing the accurate_export values in the group

2. Dividing the sum of accurate exports by the total exports in that group.

3. Multiplying the result by 100 to express the reliability as a percentage.

In [168]:
trade_df['exporter_commodity_reliability'] = (
    trade_df.groupby(['exporter', 'cmdCode'])['total_accurate_exports'].transform(lambda x: (x / trade_df.loc[x.index, 'total_exports']) * 100))

 Example of a sliced dataframe

---



In [169]:
usa_df = trade_df[trade_df['exporter'] == 'USA']
usa_df_cmdcode = usa_df[usa_df['cmdCode'] == '2710']
usa_df_cmdcode.sample(n=5)

,period,importerISO,importer,exporterISO,exporter,cmdCode,import_value,export_value,new_margin,new_import_value,accuracy_level,total_imports,total_exports,accurate_import,accurate_export,total_accurate_imports,total_accurate_exports,importer_commodity_reliability,exporter_commodity_reliability
4470674,2021,LUX,Luxembourg,USA,USA,2710,1.411662e+05,23545.0,94.03,1.327385e+05,8.332107e+01,1.565936e+09,8.493699e+10,0.0,0.0,1.530529e+09,2.923847e+10,97.738917,34.423717
4511550,2021,SVK,Slovakia,USA,USA,2710,2.305692e+06,140283.0,99.34,2.290474e+06,9.391580e+01,1.171824e+09,8.493699e+10,0.0,0.0,9.955401e+08,2.923847e+10,84.956447,34.423717
4513621,2021,ZAF,South Africa,USA,USA,2710,0.000000e+00,129357010.0,91.24,0.000000e+00,1.293570e+20,1.100000e-08,8.493699e+10,0.0,0.0,0.000000e+00,2.923847e+10,0.000000,34.423717
4501986,2021,RUS,Russian Federation,USA,USA,2710,2.383544e+07,14166235.0,92.18,2.197151e+07,4.056651e+01,1.344523e+09,8.493699e+10,0.0,0.0,4.588730e+08,2.923847e+10,34.129069,34.423717
4506071,2021,STP,Sao Tome and Principe,USA,USA,2710,1.345951e+06,0.0,91.08,1.225892e+06,1.000000e+02,3.202337e+07,8.493699e+10,0.0,0.0,0.000000e+00,2.923847e+10,0.000000,34.423717


In [170]:
 trade_df = trade_df[['period','cmdCode','importer','importerISO', 'exporter','exporterISO','export_value', 'import_value','importer_commodity_reliability', 'exporter_commodity_reliability']]

### Step 6: Based on the reliability indexes calculated get the most accurate transcation.
 **Example**

 If the reliability index for the importer is 90% and for the exporter is 75%, the function would return the importer_value since the importer's data is considered more reliable.

 By choosing the value from the partner with the higher reliability index, the function tries to achieve a more accurate representation of trade values.

In [171]:
def reconcile_trade(import_value, export_value, importer_commodity_reliability, exporter_commodity_reliability):
    """
    Reconciles trade data by considering the more reliable partner's data.
    """
    if importer_commodity_reliability >= exporter_commodity_reliability:
        return import_value  # Trust importer's data more
    else:
        return export_value  # Trust exporter's data more


In [172]:
trade_df.loc[:, 'reconciled_value'] = trade_df.apply(
    lambda x: reconcile_trade( x['import_value'],x['export_value'],x['importer_commodity_reliability'],x['exporter_commodity_reliability']),axis=1)

In [173]:
trade_df = trade_df[['period','cmdCode','exporter', 'exporterISO', 'importer', 'importerISO','reconciled_value']]

save to csv file

In [174]:
#root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Reconciled data"
#file_name= "reconciled_2021_data.csv"
#trade_df.to_csv(f'{root_folder}/{file_name}', index=False)

In [175]:
trade_df.head()

,period,cmdCode,exporter,exporterISO,importer,importerISO,reconciled_value
0,2021,9999,Afghanistan,AFG,Albania,ALB,312.741
1,2021,4202,Afghanistan,AFG,Andorra,AND,598.081
2,2021,6210,Afghanistan,AFG,Andorra,AND,199.849
3,2021,6403,Afghanistan,AFG,Andorra,AND,167.117
4,2021,8708,Afghanistan,AFG,Andorra,AND,461.344


In [176]:
# Group by exporter and sum the reconciled value
exporter_reconciled_value = trade_df.groupby('exporter')['reconciled_value'].sum()
exporter_reconciled_value.sort_values(ascending=False).reset_index().head(40)

,exporter,reconciled_value
0,China,3.257455e+12
1,USA,1.829926e+12
2,Germany,1.617260e+12
3,Japan,7.915886e+11
4,Netherlands,6.564998e+11
5,Rep. of Korea,6.443230e+11
6,Italy,6.223031e+11
7,France,5.848837e+11
8,Russian Federation,5.479781e+11
9,Mexico,4.993236e+11


In [177]:
# Group by exporter and sum the reconciled value
importer_reconciled_value = trade_df.groupby('importer')['reconciled_value'].sum()
importer_reconciled_value.sort_values(ascending=False).head(10)

,reconciled_value
importer,
USA,2.912550e+12
China,2.171041e+12
Germany,1.406923e+12
"China, Hong Kong SAR",7.452143e+11
France,7.289421e+11
Japan,7.235047e+11
Netherlands,7.097164e+11
United Kingdom,6.969099e+11
Rep. of Korea,6.058197e+11


In [178]:
total_value = trade_df['reconciled_value'].sum()
total_value

22076512538627.29

In [179]:
print(trade_df['exporter'].nunique())
print(trade_df['exporter'].nunique())

244
244


# Complexity Calculations

In [180]:
trade_df_copy = trade_df.copy()

In [181]:
threshold_country = 1500000000
threshold_HS = 500000000
# Filter out countries
df_country =  trade_df_copy.groupby('exporter')['reconciled_value'].sum()
trade_df_copy = trade_df_copy[~trade_df_copy['exporter'].isin(df_country[df_country < threshold_country].index)]

# Filter out products (HS4)
df_HS = trade_df_copy.groupby('cmdCode')['reconciled_value'].sum()
trade_df_copy = trade_df_copy[~trade_df_copy['cmdCode'].isin(df_HS[df_HS < threshold_HS].index)]

In [182]:
print(trade_df_copy['exporter'].nunique())
print(trade_df_copy['cmdCode'].nunique())

155
1024


In [183]:
comp = econci.Complexity(trade_df_copy, c='exporter', p='cmdCode', values='reconciled_value')
comp.calculate_indexes()

In [184]:
eci = comp.eci
pci = comp.pci

In [185]:
# create dataframes
eci = eci.rename(columns={0: "ECI"}).reset_index()
pci = pci.rename(columns={0: "PCI"}).reset_index()

In [186]:
# display data
print(" Highest ECIs")
eci.sort_values(by='eci',ascending=False).head(25)

 Highest ECIs


,index,exporter,eci
70,70,Japan,2.140146
106,106,"Other Asia, nes",2.083406
134,134,Switzerland,1.910716
117,117,Rep. of Korea,1.877385
54,54,Germany,1.777145
124,124,Singapore,1.713677
36,36,Czechia,1.616994
8,8,Austria,1.576992
126,126,Slovenia,1.519480
133,133,Sweden,1.503112


In [187]:
# display data
print("5 Highest PCIs")
pci.sort_values(by='pci',ascending=False).head(25)

5 Highest PCIs


,index,cmdCode,pci
301,301,3705,2.481404
826,826,8457,1.996577
196,196,2843,1.978164
813,813,8444,1.968892
853,853,8486,1.921839
315,315,3818,1.871586
300,300,3702,1.847464
744,744,8113,1.839532
302,302,3707,1.827281
790,790,8420,1.827244


In [188]:
root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/results/2021"
eci.to_csv(f'{root_folder}/eci_2021.csv', index=False)
pci.to_csv(f'{root_folder}/pci_2021.csv', index=False)